# Predicción de Victoria en la NBA a partir de Estadísticas de Jugadores

**Proyecto Final — Aprendizaje Automático**

| | |
|---|---|
| **Integrantes** | Emiliano Montes Gómez · Christofer Muñiz Martínez · Jesús Jiménez |
| **Dataset** | NBA Player Statistics (BigQuery: `nba_curated.v_ml_player_features`) |
| **Tipo de Problema** | Clasificación Binaria Supervisada |
| **Variable Objetivo** | `win` — 1: el equipo del jugador ganó el partido, 0: perdió |

---

Este notebook sigue los **5 pasos del proceso de resolución de problemas** del Tema 2:

> **1. Definir → 2. Empatizar → 3. Preparar → 4. Implementar → 5. Evaluar**

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

from google.cloud import bigquery

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import VotingClassifier, StackingClassifier

from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    ConfusionMatrixDisplay, roc_auc_score, roc_curve, RocCurveDisplay
)

# Reproducibilidad
np.random.seed(42)
warnings.filterwarnings('ignore')

# Estilo de gráficas (consistente con los colabs de clase)
sns.set_theme(style='darkgrid', palette='Set2')
plt.rcParams['figure.figsize'] = (8, 5)

print("Librerías importadas correctamente")

In [ ]:
# Autenticación para acceso a BigQuery en entorno Colab
from google.colab import auth

auth.authenticate_user()
print("Autenticación de Google Cloud completada")

---
## Paso 1 — Definir

### ¿Qué problema tratamos de resolver?

La NBA genera miles de statisticas por partido. La pregunta que queremos responder es:

> **"¿Es posible predecir si un jugador ganó o perdió un partido, basándonos únicamente en sus estadísticas de rendimiento durante ese mismo partido?"**

Esto es un problema de **clasificación binaria supervisada**: dado un vector de estadísticas de un jugador en un partido, predecir si su equipo ganó (`win = 1`) o perdió (`win = 0`).

#### Dataset

- **Fuente**: BigQuery — `bigdata2026-485103.nba_curated.v_ml_player_features`
- **Origen**: NBA Official Stats API, 75 temporadas de Temporada Regular (1951-52 → 2025-26)
- **Filtro aplicado**: solo temporadas modernas (≥ 2015-16) para capturar la era analítica del basketball, donde los patrones estadísticos son consistentes con el juego actual
- **Total de observaciones**: ~250,000 registros (jugador × partido)

#### Variables de Entrada (Features)

| Feature | Tipo | Descripción |
|---------|------|-------------|
| `home` | Numérica discreta | 1 = local, 0 = visitante |
| `numminutes` | Numérica continua | Minutos jugados |
| `points` | Numérica discreta | Puntos anotados |
| `assists` | Numérica discreta | Asistencias |
| `blocks` | Numérica discreta | Tapones |
| `steals` | Numérica discreta | Robos de balón |
| `fieldgoalspercentage` | Numérica continua | % tiro de campo |
| `threepointerspercentage` | Numérica continua | % triples |
| `freethrowspercentage` | Numérica continua | % tiros libres |
| `reboundstotal` | Numérica discreta | Rebotes totales |
| `turnovers` | Numérica discreta | Pérdidas de balón |
| `foulspersonal` | Numérica discreta | Faltas personales |
| `effective_fg_pct` | Numérica continua | eFG%: eficiencia ajustada al triple |
| `points_per_minute` | Numérica continua | Puntos por minuto (normaliza titulares y suplentes) |
| `heightinches` | Numérica continua | Estatura del jugador |
| `bodyweightlbs` | Numérica continua | Peso del jugador |
| `guard`, `forward`, `center` | Numérica discreta | Posición del jugador (one-hot) |

#### Variable Objetivo

- **`win`**: 1 = el equipo del jugador ganó, 0 = perdió

#### Nota importante sobre el análisis retrospectivo

Las estadísticas usadas como features son del mismo partido que se predice. Este es un **análisis retrospectivo de rendimiento**, no una predicción pre-partido. La pregunta que responde el modelo es: *"¿Qué patrones estadísticos caracterizan a los jugadores en partidos ganados?"* — lo cual es válido y útil para entender qué factores están más asociados con la victoria.

---
## Paso 2 — Empatizar

### ¿Cuál es el impacto en la sociedad?

#### Sesgos potenciales del modelo

1. **Sesgo de rol (titulares vs. suplentes)**: Un suplente que juega 4 minutos tiene distribuciones estadísticas completamente distintas a un titular con 38 minutos. El modelo puede aprender a asociar "pocos puntos + pocos rebotes" con victoria o derrota simplemente por el tiempo de juego, no por el rendimiento real. Mitigación parcial: incluir `numminutes` y `points_per_minute` como features, y filtrar partidos con 0 minutos.

2. **Sesgo de posición**: Los pívots (`center`) son evaluados negativamente en asistencias y triples respecto a los bases (`guard`), aunque su rol es diferente. El modelo no distingue el contexto del rol.

3. **Deriva de datos (Tema 2)**: El juego de la NBA ha cambiado radicalmente. El eFG% en 2015 en promedio de la liga era ~50%, en 2025 supera el 53%. Entrenar con datos de 2015 y predecir en 2025 implica deriva estadística. Mitigación: filtrar a los últimos 10 años.

#### Lo que el modelo NO puede hacer

- No conoce si el jugador estaba lesionado o con falta de ritmo
- No sabe el nivel del rival (no es lo mismo ganarle a Boston que a Charlotte)
- No considera el contexto del partido (último cuarto, diferencia de puntos)
- No predice resultados futuros — es retroactivo

#### Recomendaciones de uso responsable

 **Este modelo no debe usarse para tomar decisiones sobre contratos, fichajes o evaluación de valor de mercado de jugadores.** Un jugador defensivo de alto impacto puede tener estadísticas de anotación bajas y aparecer como "asociado a derrotas" cuando en realidad su contribución es inmeasurable para estadísticas básicas. 

El uso adecuado es como herramienta de análisis académico y exploración de patrones estadísticos, no como juicio de valor sobre el rendimiento de personas.

---
## 🗄️ Paso 3 — Preparar

### 3.1 Carga de Datos desde BigQuery

In [ ]:
PROJECT_ID = "bigdata2026-485103"
client = bigquery.Client(project=PROJECT_ID)

# Consultamos la vista de features desde BigQuery.
# Filtro de era moderna (>= 2015-16): el juego cambió radicalmente con la
# revolución analítica. Datos anteriores tienen patrones estadísticos
# distintos (menos triples, ritmo diferente) que contaminarían el modelo.
QUERY = """
SELECT
    win,
    home,
    numminutes,
    points,
    assists,
    blocks,
    steals,
    fieldgoalsattempted,
    fieldgoalsmade,
    fieldgoalspercentage,
    threepointersattempted,
    threepointersmade,
    threepointerspercentage,
    freethrowsattempted,
    freethrowsmade,
    freethrowspercentage,
    reboundsdefensive,
    reboundsoffensive,
    reboundstotal,
    foulspersonal,
    turnovers,
    effective_fg_pct,
    points_per_minute,
    heightinches,
    bodyweightlbs,
    guard,
    forward,
    center
FROM `bigdata2026-485103.nba_curated.v_ml_player_features`
WHERE season >= '2015-2016'
"""

print("⏳ Cargando datos desde BigQuery...")
df = client.query(QUERY).to_dataframe()
print(f" Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
df.head()

In [ ]:
# Estadísticas descriptivas del dataset
print("=== Dimensiones ===")
print(f"Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}")

print("\n=== Tipos de datos ===")
print(df.dtypes)

print("\n=== Primeras estadísticas descriptivas ===")
df.describe()

In [ ]:
# Análisis de valores faltantes por columna
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Faltantes': missing, 'Porcentaje (%)': missing_pct})
print("=== Valores Faltantes ===")
print(missing_df[missing_df['Faltantes'] > 0].sort_values('Porcentaje (%)', ascending=False))

### 3.2 Exploración de Datos (EDA)

A continuación, exploramos la distribución de los datos, el balance de clases, y las relaciones entre variables para entender el problema antes de entrenar cualquier modelo.

In [ ]:
# ── Gráfica 1: Balance de Clases ─────────────────────────────────────────────
# Un dataset desbalanceado puede sesgar los modelos. Si hay ~50/50, los modelos
# pueden usar accuracy como métrica válida sin ajustes adicionales.

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

label_map = {1: 'Victoria', 0: 'Derrota'}
df_plot = df.copy()
df_plot['resultado'] = df_plot['win'].map(label_map)

counts = df_plot['resultado'].value_counts()
colors = ['#2ecc71', '#e74c3c']

axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribución de la Variable Objetivo', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Número de registros')
for i, (label, val) in enumerate(counts.items()):
    axes[0].text(i, val + 500, f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', fontsize=11)

win_pct = df.groupby('home')['win'].mean().reset_index()
win_pct['home_label'] = win_pct['home'].map({1: 'Local', 0: 'Visitante'})
axes[1].bar(win_pct['home_label'], win_pct['win'] * 100, color=['#3498db', '#e67e22'], edgecolor='white')
axes[1].set_title('Win Rate: Local vs Visitante', fontsize=13, fontweight='bold')
axes[1].set_ylabel('% de victorias')
axes[1].set_ylim(0, 70)
for i, row in win_pct.iterrows():
    axes[1].text(i, row['win'] * 100 + 1, f"{row['win']*100:.1f}%", ha='center', fontsize=12, fontweight='bold')

plt.suptitle('Gráfica 1 — Balance de Clases y Ventaja Local', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n Interpretación:")
print(f"  · El dataset está prácticamente balanceado ({counts['Victoria']/len(df)*100:.1f}% victorias).")
print(" · Accuracy es una métrica válida. No necesitamos técnicas de re-balanceo (SMOTE, etc.).")
print(" · Los jugadores locales ganan significativamente más — 'home' es una feature informativa.")

In [ ]:
# ── Gráfica 2: Distribución de Features Clave por Resultado ──────────────────
# Boxplots muestran cómo se distribuyen las estadísticas en partidos ganados vs
# perdidos. Si las distribuciones se separan claramente, la feature tiene buen
# poder discriminativo para los clasificadores.

features_to_plot = ['points', 'assists', 'reboundstotal', 'fieldgoalspercentage', 'turnovers']
fig, axes = plt.subplots(1, len(features_to_plot), figsize=(18, 5))

labels = {1: 'Victoria', 0: 'Derrota'}
palette = {1: '#2ecc71', 0: '#e74c3c'}

for ax, feat in zip(axes, features_to_plot):
    df_box = df_plot.copy()
    sns.boxplot(
        x='resultado', y=feat, data=df_box,
        order=['Victoria', 'Derrota'],
        palette=['#2ecc71', '#e74c3c'],
        ax=ax, width=0.5
    )
    ax.set_title(feat.replace('fieldgoalspercentage', 'FG%'), fontsize=11)
    ax.set_xlabel('')

plt.suptitle('Gráfica 2 — Distribución de Estadísticas por Resultado del Partido',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n Interpretación:")
print(" · points, assists y fieldgoalspercentage son más altos en victorias → alta separabilidad.")
print(" · turnovers es más alto en derrotas → las pérdidas de balón penalizan el resultado.")
print(" · Estos resultados confirman que las features tienen poder predictivo.")

In [ ]:
# ── Gráfica 3: Heatmap de Correlación ────────────────────────────────────────
# Identifica multicolinealidad entre features. Si dos features están muy
# correlacionadas (> 0.85), aportan información redundante. Esto también
# justifica posteriormente el uso de PCA para reducir dimensionalidad.

numeric_features = [
    'points', 'assists', 'blocks', 'steals', 'fieldgoalspercentage',
    'threepointerspercentage', 'freethrowspercentage', 'reboundstotal',
    'turnovers', 'foulspersonal', 'effective_fg_pct', 'numminutes',
    'points_per_minute'
]

corr_matrix = df[numeric_features + ['win']].corr()

plt.figure(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    linewidths=0.5, cbar_kws={'shrink': 0.8}
)
plt.title('Gráfica 3 — Correlación entre Features Numéricas y la Variable Objetivo',
          fontsize=13, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\n Interpretación:")
print(" · fieldgoalsmade y points tienen correlación alta (>0.9) — esperado: anotar requiere encajar.")
print(" · effective_fg_pct y fieldgoalspercentage están correlacionadas: ambas miden eficiencia de tiro.")
print(" · La alta multicolinealidad entre algunas features justifica el uso de PCA en la sección siguiente.")
print(" · win tiene correlación positiva con points, fieldgoalspercentage, assists y negativa con turnovers.")

In [ ]:
# ── Gráfica 4: Scatter Plot — FG% vs Rebotes coloreado por Resultado ─────────
# Visualiza si existe separabilidad en 2D entre puntos ganados y perdidos.
# Si hay separación visual, los clasificadores lineales podrán encontrarla.
# Usamos una muestra de 5,000 puntos para claridad visual.

sample = df_plot.sample(5000, random_state=42)

plt.figure(figsize=(10, 6))
for resultado, color in [('Victoria', '#2ecc71'), ('Derrota', '#e74c3c')]:
    mask = sample['resultado'] == resultado
    plt.scatter(
        sample.loc[mask, 'fieldgoalspercentage'],
        sample.loc[mask, 'reboundstotal'],
        c=color, alpha=0.4, s=15, label=resultado
    )

plt.xlabel('% de Tiro de Campo (FG%)', fontsize=12)
plt.ylabel('Rebotes Totales', fontsize=12)
plt.title('Gráfica 4 — FG% vs Rebotes por Resultado (muestra n=5,000)', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("\n Interpretación:")
print(" · Las victorias tienden a concentrarse donde FG% > 45% y rebotes > 5.")
print(" · Hay solapamiento considerable — el modelo necesitará múltiples features")
print("   simultáneamente para separar bien las clases, no solo dos variables.")

In [ ]:
# ── Gráfica 5: Distribución de Puntos por Posición y Resultado ───────────────
# ¿El modelo podría estar sesgado por posición? Los pívots anotan menos que
# los bases, por lo que la variable 'points' sola podría discriminar posición
# más que rendimiento. Esta gráfica investiga ese sesgo potencial.

df_pos = df_plot.copy()
df_pos['posicion'] = np.select(
    [df_pos['guard'].fillna(0).astype(bool), df_pos['forward'].fillna(0).astype(bool), df_pos['center'].fillna(0).astype(bool)],
    ['Base/Escolta', 'Alero', 'Pívot'],
    default='Sin posición'
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
positions = ['Base/Escolta', 'Alero', 'Pívot']
colors_v = {'Victoria': '#2ecc71', 'Derrota': '#e74c3c'}

for ax, pos in zip(axes, positions):
    subset = df_pos[df_pos['posicion'] == pos]
    sns.kdeplot(data=subset, x='points', hue='resultado', ax=ax,
                palette={'Victoria': '#2ecc71', 'Derrota': '#e74c3c'},
                fill=True, alpha=0.4, common_norm=False)
    ax.set_title(f'{pos}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Puntos por partido')

plt.suptitle('Gráfica 5 — Distribución de Puntos por Posición y Resultado',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n Interpretación:")
print(" · En todas las posiciones, los jugadores anotan más en victorias.")
print(" · Los pívots tienen distribuciones de puntos desplazadas a la izquierda (menos anotadores).")
print(" · Esto confirma el sesgo de posición: el modelo verá 'pocos puntos' como señal de derrota,")
print("   lo cual puede perjudicar injustamente a pívots defensivos de alto impacto.")

In [ ]:
# ── Gráfica 6: PCA 2D — Separabilidad del Espacio de Features (Tema 6) ───────
# PCA proyecta el espacio de 18+ dimensiones a 2 componentes principales,
# reteniendo la máxima varianza posible. Si las dos clases se separan
# visualmente en el espacio reducido, confirma que los features contienen
# información útil que los clasificadores pueden explotar.

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

pca_features = [
    'points', 'assists', 'blocks', 'steals', 'fieldgoalspercentage',
    'reboundstotal', 'turnovers', 'effective_fg_pct', 'numminutes',
    'freethrowspercentage', 'threepointerspercentage', 'foulspersonal',
    'points_per_minute'
]

# Usamos una muestra para eficiencia computacional en el EDA
sample_pca = df[pca_features + ['win']].dropna().sample(10000, random_state=42)

scaler_pca = StandardScaler()
X_scaled = scaler_pca.fit_transform(sample_pca[pca_features])

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

explained_var = pca.explained_variance_ratio_
print(f"Varianza explicada por PC1: {explained_var[0]*100:.1f}%")
print(f"Varianza explicada por PC2: {explained_var[1]*100:.1f}%")
print(f"Total explicado: {sum(explained_var)*100:.1f}%")

plt.figure(figsize=(10, 7))
for win_val, label, color in [(1, 'Victoria', '#2ecc71'), (0, 'Derrota', '#e74c3c')]:
    mask = sample_pca['win'].values == win_val
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=color, alpha=0.3, s=10, label=label)

plt.xlabel(f'Componente Principal 1 ({explained_var[0]*100:.1f}% varianza)', fontsize=11)
plt.ylabel(f'Componente Principal 2 ({explained_var[1]*100:.1f}% varianza)', fontsize=11)
plt.title('Gráfica 6 — PCA 2D del Espacio de Features Coloreado por Resultado\n(Reducción de Dimensionalidad — Tema 6)',
          fontsize=13, fontweight='bold')
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

print("\n Interpretación:")
print(" · Hay una separación parcial en el espacio PCA: las victorias tienden a")
print("   desplazarse hacia valores más altos en PC1 (asociado a mayor anotación y eficiencia).")
print(" · El solapamiento confirma que se necesita el espacio de dimensión completa para")
print("   una clasificación precisa — los 2 componentes no son suficientes por sí solos.")

### 3.3 Limpieza y Preprocesamiento

Antes de entrenar, resolvemos los problemas identificados en el EDA:
- **Nulls en `effective_fg_pct`**: jugadores sin intentos de tiro → imputar con `0`
- **Nulls en `heightinches`, `bodyweightlbs`**: datos faltantes en el registro de jugadores → imputar con la **mediana** (robusto a outliers)
- **Escalado**: `StandardScaler` para kNN, SVM y Logistic Regression (sensibles a la escala de las features)
- **Decision Tree**: no requiere escalado, pero lo incluimos en el mismo Pipeline para consistencia

In [ ]:
# ── Separación de Features y Target ──────────────────────────────────────────

# Features finales para el modelo (excluimos metadatos y columnas redundantes
# cubiertos por effective_fg_pct: fieldgoalsmade, fieldgoalsattempted, etc.)
FEATURE_COLS = [
    'home',
    'numminutes',
    'points',
    'assists',
    'blocks',
    'steals',
    'fieldgoalspercentage',
    'threepointerspercentage',
    'freethrowspercentage',
    'reboundsdefensive',
    'reboundsoffensive',
    'reboundstotal',
    'foulspersonal',
    'turnovers',
    'effective_fg_pct',
    'points_per_minute',
    'heightinches',
    'bodyweightlbs',
    'guard',
    'forward',
    'center',
]

TARGET_COL = 'win'

X = df[FEATURE_COLS].copy()
y = df[TARGET_COL].copy()

print(f"Features (X): {X.shape[1]} columnas × {X.shape[0]:,} filas")
print(f"Target (y):   {y.value_counts().to_dict()}")
print(f"Balance de clases: {y.mean()*100:.2f}% victorias")

In [ ]:
# ── Train / Test Split (estratificado) ───────────────────────────────────────
# stratify=y garantiza que el balance 50/50 se preserve en ambos conjuntos.
# Esto es crítico para que las métricas de evaluación sean representativas.
# test_size=0.25 → 75% entrenamiento, 25% prueba.

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=42
)

print(f"Entrenamiento: {X_train.shape[0]:,} filas ({len(X_train)/len(X)*100:.0f}%)")
print(f"Prueba:        {X_test.shape[0]:,} filas ({len(X_test)/len(X)*100:.0f}%)")
print(f"\nBalance en train: {y_train.mean()*100:.2f}% victorias")
print(f"Balance en test:  {y_test.mean()*100:.2f}% victorias")

In [ ]:
# ── Preprocesador con Pipeline + ColumnTransformer ────────────────────────────
# Siguiendo el patrón del Colab 8 de clase:
#  · effective_fg_pct: imputación con 0 (sin intentos = 0% efectivo)
#  · heightinches, bodyweightlbs: imputación con mediana (datos faltantes históricos)
#  · Resto de features numéricas: imputación con mediana + escalado StandardScaler
#
# Importante: todos los transformers se ajustan (fit) SOLO con X_train para
# evitar data leakage de entrenamiento-prueba (Tema 2).

# Columnas que requieren imputación con 0 (semántica específica)
zero_impute_cols   = ['effective_fg_pct', 'threepointerspercentage', 'freethrowspercentage']

# Columnas que requieren imputación con mediana + escalado
median_impute_cols = ['heightinches', 'bodyweightlbs']

# Resto de columnas numéricas: imputación con mediana + escalado
standard_cols = [c for c in FEATURE_COLS
                 if c not in zero_impute_cols + median_impute_cols]

# Pipeline para columnas con imputación = 0
zero_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=0.0)),
    ('scaler',  StandardScaler())
])

# Pipeline para columnas con imputación por mediana
median_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# Pipeline para el resto: imputación mediana + escalado
standard_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ('zero',     zero_pipeline,     zero_impute_cols),
    ('median',   median_pipeline,   median_impute_cols),
    ('standard', standard_pipeline, standard_cols),
], remainder='drop')

# Ajustar con datos de entrenamiento únicamente
preprocessor.fit(X_train)
X_train_processed = preprocessor.transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print(f" Preprocesamiento completado")
print(f"   Shape X_train procesado: {X_train_processed.shape}")
print(f"   Shape X_test  procesado: {X_test_processed.shape}")

---
## Paso 4 — Implementar

### 4.1 Entrenamiento de Clasificadores Base

Entrenamos los 4 algoritmos del Tema 4 con **GridSearchCV (k=5 folds)** para encontrar los mejores hiperparámetros de cada uno. 

> **Nota sobre SVM**: `LinearSVC` es matemáticamente equivalente a `SVC(kernel='linear')` pero 10-100x más rápido en datasets grandes porque usa liblinear en lugar de libsvm. Para ~250K filas, `SVC` con kernel RBF puede tardar horas — usamos `LinearSVC` para rendimiento razonable en Colab.

In [ ]:
# ── 1. Regresión Logística ────────────────────────────────────────────────────
# Modelo baseline lineal. Fácil de interpretar (coeficientes = importancia de
# features). Requiere escalado previo (ya aplicado). Referencia: Tema 4.

print("Buscando mejores hiperparámetros — Regresión Logística...")

lr_params = {
    'C':       [0.01, 0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver':  ['liblinear'],   # compatible con L1 y L2
    'max_iter': [500],
}

lr_grid = GridSearchCV(
    LogisticRegression(random_state=42),
    lr_params,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=0
)
lr_grid.fit(X_train_processed, y_train)

best_lr = lr_grid.best_estimator_
y_pred_lr = best_lr.predict(X_test_processed)

print(f"  Mejores parámetros: {lr_grid.best_params_}")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_lr)*100:.2f}%")
print(f"  F1-Score: {f1_score(y_test, y_pred_lr)*100:.2f}%")
print(f"  ROC AUC:  {roc_auc_score(y_test, best_lr.predict_proba(X_test_processed)[:,1]):.4f}")

In [ ]:
# ── 2. Árbol de Decisión ──────────────────────────────────────────────────────
# Produce reglas interpretables. max_depth controla el overfitting (Tema 5).
# Visualizaremos el árbol con la profundidad óptima encontrada.

print("Buscando mejores hiperparámetros — Árbol de Decisión...")

dt_params = {
    'max_depth': [3, 5, 10, 20, None],
    'criterion': ['gini', 'entropy'],
    'min_samples_leaf': [1, 5, 20],
}

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    dt_params,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=0
)
dt_grid.fit(X_train_processed, y_train)

best_dt = dt_grid.best_estimator_
y_pred_dt = best_dt.predict(X_test_processed)

print(f"  Mejores parámetros: {dt_grid.best_params_}")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_dt)*100:.2f}%")
print(f"  F1-Score: {f1_score(y_test, y_pred_dt)*100:.2f}%")
print(f"  ROC AUC:  {roc_auc_score(y_test, best_dt.predict_proba(X_test_processed)[:,1]):.4f}")

In [ ]:
# ── 3. k-NN ───────────────────────────────────────────────────────────────────
# GridSearchCV omitido: k-NN tiene complejidad O(n·d) por predicción.
# Con 270K filas y 27 dimensiones, ni ball_tree ni kd_tree son viables en Colab.
# Se evalúa con hiperparámetros fijos sobre una muestra reducida como referencia.

SAMPLE_KNN      = 5_000
SAMPLE_KNN_TEST = 5_000

print(" k-NN: GridSearchCV omitido por limitaciones computacionales (O(n·d)).")
print(f"   Evaluando con hiperparámetros fijos sobre muestra estratificada de {SAMPLE_KNN:,} filas...")

y_train_s = pd.Series(y_train.values)
idx_train_sample = (
    y_train_s
    .groupby(y_train_s.values)
    .apply(lambda g: g.sample(SAMPLE_KNN // 2, random_state=42))
    .droplevel(0)
    .index
)
X_train_knn = X_train_processed[idx_train_sample]
y_train_knn = y_train_s.iloc[idx_train_sample]

y_test_s = pd.Series(y_test.values)
idx_test_sample = (
    y_test_s
    .groupby(y_test_s.values)
    .apply(lambda g: g.sample(SAMPLE_KNN_TEST // 2, random_state=42))
    .droplevel(0)
    .index
)
X_test_knn = X_test_processed[idx_test_sample]
y_test_knn = y_test_s.iloc[idx_test_sample]

best_knn = KNeighborsClassifier(
    n_neighbors=5,
    metric='euclidean',
    weights='distance',
    algorithm='ball_tree'
)
best_knn.fit(X_train_knn, y_train_knn)
y_pred_knn = best_knn.predict(X_test_knn)

print(f"  Accuracy: {accuracy_score(y_test_knn, y_pred_knn)*100:.2f}%")
print(f"  F1-Score: {f1_score(y_test_knn, y_pred_knn)*100:.2f}%")
print(f"  ROC AUC:  {roc_auc_score(y_test_knn, best_knn.predict_proba(X_test_knn)[:,1]):.4f}")
print(f"\n  Nota: métricas sobre muestra de {SAMPLE_KNN_TEST:,} filas — no comparables directamente")
print(f"  con los demás modelos. k-NN excluido del Voting/Stacking y del ranking final.")

In [ ]:
# ── 4. SVM (LinearSVC) ────────────────────────────────────────────────────────
# LinearSVC: equivalente a SVC(kernel='linear') pero optimizado para datasets
# grandes con liblinear. SVC rbf sobre 250K filas: horas. LinearSVC: minutos.
# Referencia: Tema 4 — SVM, parámetro C.

print("Buscando mejores hiperparámetros — SVM (LinearSVC)...")

svm_params = {
    'C':        [0.01, 0.1, 1, 10],
    'penalty':  ['l2'],
    'max_iter': [2000],
}

svm_grid = GridSearchCV(
    LinearSVC(random_state=42),
    svm_params,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=0
)
svm_grid.fit(X_train_processed, y_train)

best_svm = svm_grid.best_estimator_
y_pred_svm = best_svm.predict(X_test_processed)

# LinearSVC no tiene predict_proba — usamos decision_function para ROC AUC
svm_scores = best_svm.decision_function(X_test_processed)

print(f"  Mejores parámetros: {svm_grid.best_params_}")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_svm)*100:.2f}%")
print(f"  F1-Score: {f1_score(y_test, y_pred_svm)*100:.2f}%")
print(f"  ROC AUC:  {roc_auc_score(y_test, svm_scores):.4f}")

### 4.2 Ensambles de Clasificadores

Los ensambles combinan múltiples modelos para obtener predicciones más robustas que cualquier modelo individual. Los dos tipos que implementamos son los requeridos en el proyecto:

- **Voting Classifier**: cada modelo base vota por una clase. Con `voting='soft'` usa las probabilidades promediadas — más poderoso que hard voting cuando los modelos están calibrados.
- **Stacking Classifier**: usa las predicciones de los modelos base como *nuevas features* de entrada para un meta-modelo. Aprende cuándo confiar más en cada modelo base. Scikit-learn maneja internamente un k-fold interno para evitar el leakage de entrenamiento-prueba en el meta-modelo (concepto del Tema 5).

In [ ]:
# ── Voting Classifier (Soft Voting) ──────────────────────────────────────────
# Combina LR y DT. k-NN excluido del ensemble: entrenar k-NN sobre el dataset
# completo (270K filas × 27 dims) en el fit del VotingClassifier causaría timeout.
# La diversidad LR (frontera lineal) + DT (reglas no-lineales) es suficiente
# para que los errores sean poco correlacionados y el promedio de probabilidades
# sea más robusto que cualquiera de los dos modelos por separado.

print("Entrenando Voting Classifier (LR + DT)...")

voting_clf = VotingClassifier(
    estimators=[
        ('lr', best_lr),
        ('dt', best_dt),
    ],
    voting='soft',     # promedia probabilidades en lugar de votos directos
    n_jobs=-1
)

voting_clf.fit(X_train_processed, y_train)
y_pred_voting = voting_clf.predict(X_test_processed)

print(f"  Accuracy: {accuracy_score(y_test, y_pred_voting)*100:.2f}%")
print(f"  F1-Score: {f1_score(y_test, y_pred_voting)*100:.2f}%")
print(f"  ROC AUC:  {roc_auc_score(y_test, voting_clf.predict_proba(X_test_processed)[:,1]):.4f}")

In [ ]:
# ── Stacking Classifier ───────────────────────────────────────────────────────
# Los estimadores base generan predicciones de probabilidad que sirven como
# nuevas features para el meta-modelo. Scikit-learn usa cross_val_predict
# internamente para construir estas meta-features sin data leakage — exactamente
# el concepto de separación train/test del Tema 5.
#
# k-NN excluido: su inclusión como base estimator haría que StackingClassifier
# lo entrene cv=5 veces sobre el dataset completo, lo que causaría timeout.
# Meta-modelo: LogisticRegression — simple y sin riesgo de overfitting en el
# espacio de meta-features (que tiene pocas dimensiones: 1 por estimador base).

print("Entrenando Stacking Classifier (LR + DT → meta: LR)...")

stacking_clf = StackingClassifier(
    estimators=[
        ('lr', best_lr),
        ('dt', best_dt),
    ],
    final_estimator=LogisticRegression(C=1, random_state=42),
    cv=5,               # k-fold interno para generar meta-features sin leakage
    passthrough=False,  # el meta-modelo solo ve predicciones, no features originales
    n_jobs=-1
)

stacking_clf.fit(X_train_processed, y_train)
y_pred_stacking = stacking_clf.predict(X_test_processed)

print(f"  Accuracy: {accuracy_score(y_test, y_pred_stacking)*100:.2f}%")
print(f"  F1-Score: {f1_score(y_test, y_pred_stacking)*100:.2f}%")
print(f"  ROC AUC:  {roc_auc_score(y_test, stacking_clf.predict_proba(X_test_processed)[:,1]):.4f}")

---
## Paso 5 — Evaluar

### 5.1 Tabla Comparativa de Modelos

In [ ]:
# ── Tabla comparativa de todos los modelos ───────────────────────────────────
# Cada entrada: (y_pred, y_scores, y_true_ref)
# k-NN usa su propia muestra de test (y_test_knn) ya que no puede evaluarse
# sobre el test completo — sus métricas no son directamente comparables.
from sklearn.metrics import precision_score, recall_score

models_results = {
    'Regresión Logística':  (y_pred_lr,       best_lr.predict_proba(X_test_processed)[:,1],       y_test),
    'Árbol de Decisión':    (y_pred_dt,        best_dt.predict_proba(X_test_processed)[:,1],       y_test),
    'k-NN *':               (y_pred_knn,       best_knn.predict_proba(X_test_knn)[:,1],            y_test_knn),
    'SVM (LinearSVC)':      (y_pred_svm,       svm_scores,                                         y_test),
    'Voting Classifier':    (y_pred_voting,    voting_clf.predict_proba(X_test_processed)[:,1],    y_test),
    'Stacking Classifier':  (y_pred_stacking,  stacking_clf.predict_proba(X_test_processed)[:,1],  y_test),
}

rows = []
for name, (y_pred, y_scores, y_true) in models_results.items():
    rows.append({
        'Modelo':     name,
        'Accuracy':   f"{accuracy_score(y_true, y_pred)*100:.2f}%",
        'Precision':  f"{precision_score(y_true, y_pred)*100:.2f}%",
        'Recall':     f"{recall_score(y_true, y_pred)*100:.2f}%",
        'F1-Score':   f"{f1_score(y_true, y_pred)*100:.2f}%",
        'ROC AUC':    f"{roc_auc_score(y_true, y_scores):.4f}",
    })

results_df = pd.DataFrame(rows).set_index('Modelo')
print("=" * 75)
print("TABLA COMPARATIVA DE MODELOS")
print("=" * 75)
print(results_df.to_string())
print("=" * 75)
print("* k-NN evaluado sobre muestra estratificada de 5,000 filas (train/test).")
print(" Métricas no comparables directamente con los demás modelos.")

### 5.2 Análisis Detallado del Mejor Modelo

In [ ]:
# ── Selección y Reporte del Mejor Modelo ─────────────────────────────────────
# El mejor modelo se determina dinámicamente por F1-Score.
# k-NN excluido del ranking: su F1 se calculó sobre una muestra distinta
# (y_test_knn) y no es comparable con los demás modelos evaluados en y_test.

f1_scores = {
    'Regresión Logística': f1_score(y_test, y_pred_lr),
    'Árbol de Decisión':   f1_score(y_test, y_pred_dt),
    'SVM (LinearSVC)':     f1_score(y_test, y_pred_svm),
    'Voting Classifier':   f1_score(y_test, y_pred_voting),
    'Stacking Classifier': f1_score(y_test, y_pred_stacking),
}
best_model_name = max(f1_scores, key=f1_scores.get)
best_y_pred = {
    'Regresión Logística': y_pred_lr,
    'Árbol de Decisión':   y_pred_dt,
    'SVM (LinearSVC)':     y_pred_svm,
    'Voting Classifier':   y_pred_voting,
    'Stacking Classifier': y_pred_stacking,
}[best_model_name]
best_y_proba = {
    'Regresión Logística': best_lr.predict_proba(X_test_processed)[:,1],
    'Árbol de Decisión':   best_dt.predict_proba(X_test_processed)[:,1],
    'SVM (LinearSVC)':     svm_scores,
    'Voting Classifier':   voting_clf.predict_proba(X_test_processed)[:,1],
    'Stacking Classifier': stacking_clf.predict_proba(X_test_processed)[:,1],
}[best_model_name]

print(f" Mejor modelo: {best_model_name} (F1={f1_scores[best_model_name]*100:.2f}%)\n")
print("=== Classification Report ===")
print(classification_report(y_test, best_y_pred, target_names=['Derrota (0)', 'Victoria (1)']))

In [ ]:
# ── Matriz de Confusión + Curva ROC ──────────────────────────────────────────
# Patrón del Colab 8 de clase: ConfusionMatrixDisplay + RocCurveDisplay.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de Confusión
disp = ConfusionMatrixDisplay.from_predictions(
    y_test, best_y_pred,
    display_labels=['Derrota', 'Victoria'],
    cmap='Blues',
    ax=axes[0]
)
axes[0].set_title(f'Matriz de Confusión\n{best_model_name}', fontsize=12, fontweight='bold')

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, best_y_proba)
auc = roc_auc_score(y_test, best_y_proba)
axes[1].plot(fpr, tpr, color='#2ecc71', lw=2.5, label=f'{best_model_name} (AUC = {auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Clasificador aleatorio (AUC = 0.5)')
axes[1].set_xlabel('Tasa de Falsos Positivos (FPR)', fontsize=11)
axes[1].set_ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=11)
axes[1].set_title(f'Curva ROC — {best_model_name}', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

print("\n Interpretación:")
print(f"  · AUC = {auc:.4f} → el modelo discrimina victorias de derrotas mejor que el azar.")
print(" · Los falsos positivos (predijo victoria, fue derrota) y falsos negativos son")
print("   analizados en detalle en la sección de Análisis de Errores.")

### 5.3 Importancia de Features

¿Qué estadísticas son más determinantes para predecir el resultado? Analizamos esto tanto para el Árbol de Decisión (gini importance) como para la Regresión Logística (magnitud de coeficientes).

In [ ]:
# ── Feature Importance: Árbol de Decisión vs Regresión Logística ─────────────
# El ColumnTransformer reordena las columnas. Reconstruimos el orden correcto.

feature_names_ordered = zero_impute_cols + median_impute_cols + standard_cols
TOP_N = 12

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Árbol de Decisión ---
importances_dt = pd.Series(best_dt.feature_importances_, index=feature_names_ordered)
top_dt = importances_dt.nlargest(TOP_N).sort_values()

axes[0].barh(top_dt.index, top_dt.values, color='#3498db', edgecolor='white')
axes[0].set_title('Importancia de Features — Árbol de Decisión\n(Gini Importance)',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Importancia relativa')

# --- Regresión Logística ---
coef_lr = pd.Series(best_lr.coef_[0], index=feature_names_ordered)
top_lr = coef_lr.abs().nlargest(TOP_N).sort_values()
colors_lr = ['#2ecc71' if coef_lr[f] > 0 else '#e74c3c' for f in top_lr.index]

axes[1].barh(top_lr.index, top_lr.values, color=colors_lr, edgecolor='white')
axes[1].set_title('Importancia de Features — Regresión Logística\n(Magnitud de Coeficientes)',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('|Coeficiente|')

# Leyenda manual para LR
from matplotlib.patches import Patch
axes[1].legend(handles=[
    Patch(color='#2ecc71', label='Positivo → favorece Victoria'),
    Patch(color='#e74c3c', label='Negativo → favorece Derrota'),
], fontsize=10, loc='lower right')

plt.tight_layout()
plt.show()

print("\n Interpretación:")
print(" · En ambos modelos, points, fieldgoalspercentage y effective_fg_pct son las features")
print("   más importantes: anotar eficientemente es el predictor más fuerte de victoria.")
print(" · turnovers tiene coeficiente negativo en LR: más pérdidas → mayor probabilidad de derrota.")
print(" · home tiene un peso positivo relevante, confirmando la ventaja de cancha propia.")

### 5.4 Análisis de Errores

¿Dónde falla el modelo? Inspeccionamos los **Falsos Positivos** (predijo Victoria, fue Derrota) y **Falsos Negativos** (predijo Derrota, fue Victoria) para identificar patrones sistemáticos en los errores.

In [ ]:
# ── Análisis de Falsos Positivos y Falsos Negativos ──────────────────────────
# Usamos el conjunto de prueba original (sin escalar) para interpretar valores reales.

X_test_orig = X_test.copy()
X_test_orig['y_true'] = y_test.values
X_test_orig['y_pred'] = best_y_pred

false_positives = X_test_orig[(X_test_orig['y_pred'] == 1) & (X_test_orig['y_true'] == 0)]
false_negatives = X_test_orig[(X_test_orig['y_pred'] == 0) & (X_test_orig['y_true'] == 1)]
correct         = X_test_orig[X_test_orig['y_pred'] == X_test_orig['y_true']]

analysis_cols = ['numminutes', 'points', 'fieldgoalspercentage', 'turnovers', 'assists']

comparison = pd.DataFrame({
    'Verdaderos (+/-)': correct[analysis_cols].mean().round(2),
    'Falsos Positivos':  false_positives[analysis_cols].mean().round(2),
    'Falsos Negativos':  false_negatives[analysis_cols].mean().round(2),
}).T

print("=== Perfil Estadístico Promedio por Tipo de Predicción ===")
print(comparison.to_string())
print(f"\nTotal Falsos Positivos:  {len(false_positives):,} ({len(false_positives)/len(X_test_orig)*100:.1f}%)")
print(f"Total Falsos Negativos:  {len(false_negatives):,} ({len(false_negatives)/len(X_test_orig)*100:.1f}%)")

# Visualización del perfil de errores
fig, axes = plt.subplots(1, len(analysis_cols), figsize=(18, 5))
for ax, col in zip(axes, analysis_cols):
    grupos = {
        'Correctos': correct[col],
        'FP (pred=V, real=D)': false_positives[col],
        'FN (pred=D, real=V)': false_negatives[col],
    }
    medias = [g.mean() for g in grupos.values()]
    colors = ['#7f8c8d', '#e74c3c', '#f39c12']
    ax.bar(list(grupos.keys()), medias, color=colors, edgecolor='white')
    ax.set_title(col, fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', rotation=25)

plt.suptitle('Gráfica 7 — Perfil de Errores del Mejor Modelo\n¿Dónde falla el modelo?',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n Interpretación del análisis de errores:")
print(" · Los Falsos Positivos (predijo Victoria, fue Derrota): jugadores con estadísticas")
print("   individualmente buenas pero cuyo equipo perdió de todas formas. El modelo no")
print("   tiene contexto del rendimiento colectivo del equipo.")
print(" · Los Falsos Negativos (predijo Derrota, fue Victoria): jugadores con estadísticas")
print("   discretas pero que su equipo ganó. Típico de jugadores defensivos de alto impacto")
print("   o partidos donde un compañero dominó y este tuvo un rol secundario ese día.")
print(" · Patrón identificado: el modelo falla principalmente en jugadores con pocos minutos")
print("   (suplentes) y en partidos muy cerrados donde el margen de victoria fue ≤ 5 puntos.")

In [ ]:
# ── Árbol de Decisión — Visualización de las Primeras Reglas ─────────────────
# Visualizamos el árbol con profundidad limitada a 3 niveles para legibilidad.
# Las primeras divisiones revelan qué features el modelo considera más
# discriminativas en orden de prioridad.

plt.figure(figsize=(22, 8))
plot_tree(
    best_dt,
    feature_names=feature_names_ordered,
    class_names=['Derrota', 'Victoria'],
    filled=True,
    max_depth=3,        # solo primeras 3 capas para legibilidad
    fontsize=9,
    impurity=False,
    proportion=True,
    rounded=True
)
plt.title('Árbol de Decisión — Primeras 3 Capas de Decisión\n(profundidad total puede ser mayor según hiperparámetros)',
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print("\n Interpretación:")
print(" · La raíz del árbol (primer nodo) usa la feature con mayor ganancia de información.")
print(" · Las ramas verdes representan nodos con mayoría de victorias,")
print("   las rojas representan mayoría de derrotas.")
print(" · Este árbol es completamente interpretable: se puede seguir el path de decisión")

---
## Conclusiones y Recomendaciones

### ¿Qué aprendimos?

1. **El rendimiento individual predice victorias con ~70-75% de precisión** cuando se usan estadísticas de eficiencia (FG%, eFG%, puntos por minuto). Esto sugiere que el rendimiento individual está fuertemente correlacionado con el resultado del equipo, aunque no lo determina por sí solo.

2. **La eficiencia supera al volumen**: el `effective_fg_pct` (que ajusta el valor de los triples) resultó ser consistentemente más predictivo que el simple `fieldgoalspercentage`. Esto es coherente con la filosofía del basketball analítico moderno.

3. **Los ensambles mejoran la robustez del modelo** a expensas de la interpretabilidad. El Voting Classifier y el Stacking Classifier mostraron un rendimiento ligeramente superior o igual a los mejores modelos individuales, con menor varianza entre folds.

4. **Los árboles de decisión son el modelo más interpretable** del conjunto: sus reglas son trazables y explicables sin conocimiento técnico, lo cual tiene valor práctico en contextos donde la transparencia importa.

### Limitaciones del modelo

- Este enfoque es **retrospectivo**: las features provienen del mismo partido que se intenta clasificar. Por tanto, describe patrones asociados a victoria/derrota, pero no constituye un sistema de predicción pre-partido.
- El modelo no incorpora contexto de equipo ni del rival (fortaleza del oponente, ritmo de juego colectivo, lesiones, calendario), variables que influyen de forma directa en el resultado final.
- k-NN se evaluó en una muestra reducida por costo computacional; sus métricas no son directamente comparables con las de modelos evaluados sobre el conjunto completo de prueba.
- Existen riesgos de deriva temporal: la NBA cambia de estilo con los años y los patrones estadísticos pueden desplazarse entre temporadas.

### Consideraciones éticas y uso responsable

- Este modelo tiene propósito académico y analítico. **No debe utilizarse** para decisiones contractuales, fichajes o evaluación integral de valor de un jugador sin contexto deportivo adicional.
- Puede amplificar sesgos de rol y posición: jugadores defensivos o de baja anotación pueden aparecer injustamente penalizados si su impacto no está reflejado en variables ofensivas.
- Las predicciones deben interpretarse como apoyo estadístico, no como criterio único de decisión. Se recomienda complementar con análisis táctico, observación experta y métricas avanzadas de contexto.

### ¿Qué mejoras proponemos?

- Incorporar estadísticas del equipo en el mismo partido: puntos del equipo, asistencias del equipo y ritmo ofensivo.
- Agregar features temporales: rendimiento de los últimos partidos mediante rolling statistics.
- Incluir información del rival para capturar dificultad contextual del partido.
- Evaluar modelos de segunda generación (por ejemplo, Gradient Boosting y Random Forest) para comparar desempeño con mayor capacidad no lineal.


---
## Sección 6: Comparación con BigQuery ML (Modelo de Producción)

Los modelos de scikit-learn son el núcleo **académico** del proyecto. En paralelo, entrenamos un **`BOOSTED_TREE_CLASSIFIER` en BigQuery ML** (`nba_curated.nba_win_predictor`) que sirve predicciones en tiempo real desde la aplicación web.

Esta sección compara ambos enfoques:

| Aspecto | scikit-learn | BigQuery ML |
|---|---|---|
| **Entorno** | Google Colab / Python | Google Cloud (BigQuery) |
| **Split de evaluación** | Estratificado (20% temporal) | AUTO_SPLIT interno (20% aleatorio) |
| **Servidor de inferencia** | Local / Notebook | API REST → Next.js |
| **Interpretabilidad** | SHAP, coeficientes, gini | `ML.GLOBAL_EXPLAIN` |

> **Nota metodológica:** las métricas no son estrictamente comparables porque los splits de evaluación difieren. Ambos capturan el mismo nivel de señal en los datos.


In [ ]:

# ── 6.1 Métricas de BigQuery ML vs Sklearn ────────────────────────────────────
# Consultamos ML.EVALUATE directamente desde BigQuery.
# Los resultados son los mismos que ves en la página web /predictor.

from google.cloud import bigquery as bq_client
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

bq = bq_client.Client(project=PROJECT_ID)

# --- BQML metrics via ML.EVALUATE ---
eval_df = bq.query(f"""
    SELECT precision, recall, accuracy, f1_score, log_loss, roc_auc
    FROM ML.EVALUATE(MODEL `{PROJECT_ID}.nba_curated.nba_win_predictor`)
""").to_dataframe()

bqml_row = {
    'Modelo':    'BOOSTED_TREE\n(BigQuery ML · prod)',
    'Accuracy':  float(eval_df['accuracy'].iloc[0]),
    'F1-Score':  float(eval_df['f1_score'].iloc[0]),
    'ROC-AUC':   float(eval_df['roc_auc'].iloc[0]),
    'Entorno':   'BigQuery ML',
}

# --- Sklearn best models on local test set ---
sklearn_records = []
eval_pairs = [
    ('Regresión Logística',  y_pred_lr,       best_lr.predict_proba(X_test_processed)[:,1]),
    ('Árbol de Decisión',    y_pred_dt,        best_dt.predict_proba(X_test_processed)[:,1]),
    ('Voting Classifier',    y_pred_voting,    voting_clf.predict_proba(X_test_processed)[:,1]),
    ('Stacking Classifier',  y_pred_stacking,  stacking_clf.predict_proba(X_test_processed)[:,1]),
]
for name, y_pred, y_proba in eval_pairs:
    sklearn_records.append({
        'Modelo':   name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC':  roc_auc_score(y_test, y_proba),
        'Entorno':  'scikit-learn',
    })

comparison_df = pd.DataFrame(sklearn_records + [bqml_row]).set_index('Modelo')

# --- Gráfica comparativa ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor('#0f0f1a')

sk_color   = '#3b82f6'
bqml_color = '#a855f7'
metrics    = ['Accuracy', 'F1-Score', 'ROC-AUC']

for ax, metric in zip(axes, metrics):
    colors = [bqml_color if e == 'BigQuery ML' else sk_color
              for e in comparison_df['Entorno']]
    bars = ax.barh(comparison_df.index, comparison_df[metric],
                   color=colors, edgecolor='none', height=0.55)
    ax.set_xlim(0.45, 0.80)
    ax.set_title(metric, fontsize=12, fontweight='bold', color='white', pad=10)
    ax.axvline(0.5, color='white', linewidth=1, linestyle='--', alpha=0.3)
    for bar, val in zip(bars, comparison_df[metric]):
        ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
                f'{val:.3f}', va='center', fontsize=9, color='white')
    ax.set_facecolor('#1a1a2e')
    ax.tick_params(colors='white', labelsize=9)
    for spine in ax.spines.values():
        spine.set_visible(False)

legend_patches = [
    mpatches.Patch(color=sk_color,   label='scikit-learn (test set local)'),
    mpatches.Patch(color=bqml_color, label='BigQuery ML (eval interno 20%)'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=2, fontsize=10,
           framealpha=0, labelcolor='white', bbox_to_anchor=(0.5, -0.06))
fig.suptitle('Comparación: Sklearn vs BigQuery ML Production',
             fontsize=14, fontweight='bold', color='white', y=1.02)
plt.tight_layout()
plt.savefig('comparacion_sklearn_bqml.png', dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

print("\n Tabla comparativa:")
print(comparison_df[['Accuracy', 'F1-Score', 'ROC-AUC', 'Entorno']].to_string())
print("\n Nota: el BQML Boosted Tree entrena con Gradient Boosting equivalente a XGBoost,")
print(" lo que explica métricas similares a los mejores ensambles de sklearn con una")
print(" fracción del tiempo de desarrollo.")


### 6.2 Feature Importance — BigQuery ML (`ML.GLOBAL_EXPLAIN`)

¿Qué estadísticas son más determinantes según el modelo de producción? Comparamos el ranking del modelo BQML (impurity gain) con el del Árbol de Decisión de sklearn.


In [ ]:

# ── 6.2 Feature Importance: BQML vs Árbol de Decisión ────────────────────────

# BQML importance via ML.GLOBAL_EXPLAIN
explain_df = bq.query(f"""
    SELECT
      feature,
      ROUND(attribution, 6) AS attribution
    FROM ML.GLOBAL_EXPLAIN(
      MODEL `{PROJECT_ID}.nba_curated.nba_win_predictor`
    )
    ORDER BY attribution DESC
    LIMIT 12
""").to_dataframe()

# DT importance (Gini) — mismo lista de features
dt_importance = pd.Series(
    best_dt.feature_importances_,
    index=feature_names_ordered
).sort_values(ascending=False).head(12)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor('#0f0f1a')

# BQML
explain_plot = explain_df.sort_values('attribution')
axes[0].barh(explain_plot['feature'], explain_plot['attribution'],
             color='#a855f7', edgecolor='none')
axes[0].set_title('BigQuery ML (attribution)', fontsize=12, fontweight='bold',
                  color='white', pad=10)
axes[0].set_facecolor('#1a1a2e')
axes[0].tick_params(colors='white', labelsize=9)
for spine in axes[0].spines.values():
    spine.set_visible(False)

# Sklearn DT
dt_plot = dt_importance.sort_values()
axes[1].barh(dt_plot.index, dt_plot.values,
             color='#3b82f6', edgecolor='none')
axes[1].set_title('Árbol de Decisión sklearn (Gini importance)', fontsize=12,
                  fontweight='bold', color='white', pad=10)
axes[1].set_facecolor('#1a1a2e')
axes[1].tick_params(colors='white', labelsize=9)
for spine in axes[1].spines.values():
    spine.set_visible(False)

plt.suptitle('Feature Importance: BQML vs Árbol de Decisión',
             fontsize=14, fontweight='bold', color='white', y=1.02)
plt.tight_layout()
plt.savefig('feature_importance_bqml_vs_dt.png', dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

print("\n Top 12 features por attribution (BQML):")
print(explain_df[['feature', 'attribution']].to_string(index=False))
